In [7]:
import os
files = os.listdir(r"\\민경모-pc\출하증원본")

print(files)


['~$NEW출하증161117.xlsm', '~$NEW출하증_2020.xlsm', '~$NEW출하증_2020_원본 (1).xlsm', '~$NEW출하증_2020_원본 (2).xlsm', '~$NEW출하증_2020_원본.xlsm', '~$출하증 2023 원본(2024.10.12).xlsm', '~$출하증 2023 원본(2025).xlsm', '~$출하증 2023 원본.xlsm', '새 폴더', '출하증 2023 원본(2025).xlsm']


In [25]:
"""
load_stock_move.py
당월자료관리 엑셀의 출하자료/화성출하 시트를 읽어서
sales.stock_move 테이블에 INSERT합니다.

실행:
    python load_stock_move.py
"""
import pandas as pd

import psycopg2
from openpyxl import load_workbook

# ── 설정 ──────────────────────────────────────────────────────
DB = {
    "host":     "158.247.208.195",
    "port":     5432,
    "dbname":   "db_hwaseong",
    "user":     "admin",
    "password": "1234",
}

#출하증 path \\민경모-pc\출하증원본\출하증 2023 원본(2025).xlsx

# 파일 목록 — 월별 파일 여러 개 추가 가능
EXCEL_PATH = r"\\민경모-pc\출하증원본\출하증 2023 원본(2025).xlsm"
df = pd.read_excel(EXCEL_PATH, sheet_name="출하자료", 
                   usecols=['No', '일자', '거래처', '납품처', '모  델', '출하수'])

# ─────────────────────────────────────────────────────────────
# 일자 조정
df = df[df['일자'] >= "2026-04-01"]



In [23]:
conn = psycopg2.connect(**DB)
partner_df = pd.read_sql("SELECT id, name FROM sales.res_partner", conn)
partner_df

C:\Users\user\AppData\Local\Temp\ipykernel_22520\2732333204.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  partner_df = pd.read_sql("SELECT id, name FROM sales.res_partner", conn)


,id,name
0,1,CJ대한통운
1,2,KEC
2,3,LG서브원
3,4,SK매직
4,5,㈜진보
...,...,...
123,124,알파어셈블리솔루션즈코리아/시흥
124,125,케이듀/아이스팩
125,126,코웨이/유구공장
126,127,코웨이/인천공장


In [28]:
df = df.merge(partner_df, left_on='거래처', right_on='name', how='left')
df = df.rename(columns={'id': 'partner_id'})
df = df.drop(columns=['name'])
df

,No,일자,거래처,납품처,모 델,출하수,partner_id
0,9,2026-04-01,알비코리아,알비코리아,디럭스플러스,300,58.0
1,10,2026-04-01,캐논코리아,"2층,E/V앞",T499-SET#3,464,98.0
2,11,2026-04-01,오비오,오비오,CHP-5730-FRONT,1000,68.0
3,11,2026-04-01,오비오,오비오,CHP-5730-LOWER,1000,68.0
4,11,2026-04-01,오비오,오비오,CHP-5730-REAR,1000,68.0
...,...,...,...,...,...,...,...
936,22,2026-04-22,코웨이/유구공장,유구공장,프라임아이스맥스-UP,216,126.0
937,22,2026-04-22,코웨이/유구공장,유구공장,CHPI-7440N-LOW,120,126.0
938,22,2026-04-22,코웨이/유구공장,유구공장,CHPI-7440N-UP,120,126.0
939,22,2026-04-22,코웨이/유구공장,유구공장,프라임아이스맥스-LOW,216,126.0


In [29]:
df = df.merge(partner_df, left_on='납품처', right_on='name', how='left')
df = df.rename(columns={'id': 'delivery_id'})
df = df.drop(columns=['name'])

df

,No,일자,거래처,납품처,모 델,출하수,partner_id,delivery_id
0,9,2026-04-01,알비코리아,알비코리아,디럭스플러스,300,58.0,58.0
1,10,2026-04-01,캐논코리아,"2층,E/V앞",T499-SET#3,464,98.0,NaN
2,11,2026-04-01,오비오,오비오,CHP-5730-FRONT,1000,68.0,68.0
3,11,2026-04-01,오비오,오비오,CHP-5730-LOWER,1000,68.0,68.0
4,11,2026-04-01,오비오,오비오,CHP-5730-REAR,1000,68.0,68.0
...,...,...,...,...,...,...,...,...
936,22,2026-04-22,코웨이/유구공장,유구공장,프라임아이스맥스-UP,216,126.0,NaN
937,22,2026-04-22,코웨이/유구공장,유구공장,CHPI-7440N-LOW,120,126.0,NaN
938,22,2026-04-22,코웨이/유구공장,유구공장,CHPI-7440N-UP,120,126.0,NaN
939,22,2026-04-22,코웨이/유구공장,유구공장,프라임아이스맥스-LOW,216,126.0,NaN


In [ ]:

def load_partner_map(conn) -> dict[str, int]:
    """거래처명 → id 매핑"""
    with conn.cursor() as cur:
        cur.execute("SELECT id, name FROM sales.res_partner")
        return {name.strip(): pid for pid, name in cur.fetchall()}


def load_product_map(conn) -> dict[str, int]:
    """모델코드 → id 매핑"""
    with conn.cursor() as cur:
        cur.execute("SELECT id, code FROM sales.product")
        return {code.strip(): pid for pid, code in cur.fetchall()}


def read_sheet(path: str, sheet_name: str) -> list[dict]:
    """
    출하자료/화성출하 시트 읽기
    컬럼 순서: 순번, 일자, 거래처, 납품처, 모델, 생산수, 출하수, 차량No, ...
    """
    wb = load_workbook(path, read_only=True, data_only=True)

    if sheet_name not in wb.sheetnames:
        print(f"  ⚠ 시트 없음: {sheet_name}")
        return []

    ws = wb[sheet_name]
    rows = list(ws.iter_rows(min_row=2, values_only=True))

    result = []
    for r in rows:
        # 순번 없으면 스킵
        if not r[0]:
            continue

        # 일자
        raw_date = r[1]
        if raw_date is None:
            continue
        shipped_date = raw_date.date() if hasattr(raw_date, 'date') else raw_date

        # 거래처/납품처
        partner_name  = str(r[2]).strip() if r[2] else None
        delivery_name = str(r[3]).strip() if r[3] else None

        # 모델
        product_code = str(r[4]).strip() if r[4] else None

        # 출하수 (인덱스 6)
        shipped_qty = int(r[6]) if r[6] and str(r[6]).isdigit() else None
        if shipped_qty is None and r[6]:
            try:
                shipped_qty = int(float(str(r[6])))
            except:
                shipped_qty = None

        # 차량No (인덱스 7)
        vehicle = str(r[7]).strip() if r[7] else None

        result.append({
            "shipped_date":   shipped_date,
            "partner_name":   partner_name,
            "delivery_name":  delivery_name,
            "product_code":   product_code,
            "shipped_qty":    shipped_qty,
            "vehicle":        vehicle,
        })

    return result


def insert_stock_moves(conn, records: list[dict],
                       partner_map: dict, product_map: dict) -> tuple[int, int]:
    """
    INSERT 실행. (성공건수, 매칭실패건수) 반환
    """
    sql = """
        INSERT INTO sales.stock_move
            (shipped_date, partner_id, delivery_id, product_id, product_code, shipped_qty, vehicle)
        VALUES
            (%(shipped_date)s, %(partner_id)s, %(delivery_id)s, %(product_id)s,
             %(product_code)s, %(shipped_qty)s, %(vehicle)s)
    """

    success = 0
    no_match = 0

    with conn.cursor() as cur:
        for rec in records:
            partner_id  = partner_map.get(rec["partner_name"])
            delivery_id = partner_map.get(rec["delivery_name"])
            product_id  = product_map.get(rec["product_code"])

            if not partner_id:
                no_match += 1

            cur.execute(sql, {
                "shipped_date":  rec["shipped_date"],
                "partner_id":    partner_id,
                "delivery_id":   delivery_id,
                "product_id":    product_id,
                "product_code":  rec["product_code"],
                "shipped_qty":   rec["shipped_qty"],
                "vehicle":       rec["vehicle"],
            })
            success += 1

    conn.commit()
    return success, no_match




DB 연결 중...
거래처 128개 / 모델 210개 로드

=== April_stockMove.xlsx ===
  [S] 읽는 중...
  ⚠ 시트 없음: S
  [S] 0행 발견
  [h] 읽는 중...
  ⚠ 시트 없음: h
  [h] 0행 발견
  [e] 읽는 중...
  ⚠ 시트 없음: e
  [e] 0행 발견
  [e] 읽는 중...
  ⚠ 시트 없음: e
  [e] 0행 발견
  [t] 읽는 중...
  ⚠ 시트 없음: t
  [t] 0행 발견
  [1] 읽는 중...
  ⚠ 시트 없음: 1
  [1] 0행 발견

✓ 완료! 총 0건 / 거래처 미매칭: 0건


In [3]:
conn.commit()

NameError: name 'conn' is not defined